In [ ]:
# 1. Mount
from google.colab import drive
drive.mount("/content/drive")

# 2. Check what survived
!ls /content/oulad/ 2>/dev/null || echo "OULAD CSVs missing"
!ls /content/drive/MyDrive/cinque_oulad/*.parquet 2>/dev/null || echo "parquet missing"

Mounted at /content/drive
OULAD CSVs missing
/content/drive/MyDrive/cinque_oulad/oulad_engagement.parquet
/content/drive/MyDrive/cinque_oulad/oulad_merged.parquet
/content/drive/MyDrive/cinque_oulad/oulad_merged_pct.parquet


In [ ]:
!pip install kaggle -q
from google.colab import files
files.upload()          # kaggle.json
!mkdir -p ~/.kaggle && cp kaggle*.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d anlgrbz/student-demographics-online-education-dataoulad -p /content --unzip
!mkdir -p /content/oulad && mv /content/*.csv /content/oulad/ 2>/dev/null
!ls /content/oulad

Saving kaggle (1).json to kaggle (1).json
Dataset URL: https://www.kaggle.com/datasets/anlgrbz/student-demographics-online-education-dataoulad
License(s): Attribution 4.0 International (CC BY 4.0)
100% 42.2M/42.2M [00:00<00:00, 223MB/s]

assessments.csv  studentAssessment.csv	studentRegistration.csv  vle.csv
courses.csv	 studentInfo.csv	studentVle.csv


In [ ]:
!pip install -q --upgrade imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.1/236.1 kB 5.4 MB/s eta 0:00:00


In [6]:
import sklearn, imblearn, xgboost
print("sklearn :", sklearn.__version__)
print("imblearn:", imblearn.__version__)
print("xgboost :", xgboost.__version__)

sklearn : 1.8.0
imblearn: 0.14.2
xgboost : 3.3.0


In [7]:
# TIER 3 — STAGE 1: Zero-shot transfer, Source -> OULAD
# Option A: retrain Source on the transferable 9-feature subset,
#           then apply it cold to OULAD. No retraining on OULAD.
# ============================================================
import joblib, numpy as np, pandas as pd, re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy.stats import rankdata

Source = "/content/drive/MyDrive/model_ready_dataset_1.xlsx"
OUT   = "/content/drive/MyDrive/cinque_oulad"
SEEDS = [42, 0, 123, 2024, 7]
BEST  = dict(max_depth=8, n_estimators=300, learning_rate=0.1, subsample=0.8)

def pct(s):                       # within-dataset percentile rank
    a = np.asarray(s, float); r = rankdata(a, method="average")
    return 100.0*(r-1)/(len(r)-1) if len(r) > 1 else np.zeros_like(a)

In [16]:
# ---------- 1. Source side ----------
kn = pd.read_excel(Source)
kn["y"] = (kn["at_risk"] == 0).astype(int)          # at-risk coded 0

K = pd.DataFrame({
    "eng_views_pct" : pct(kn["n_module_views"]),
    "eng_sess_pct"  : pct(kn["n_logins"]),
    "eng_span_pct"  : pct(kn["engagement_span_days"]),
    "score_sd"      : kn["std_module_score"],
    "score_mean"    : kn["total_score"] / 6.0,       # 6 modules -> per-module mean
    "score_improve" : kn["score_improvement"],
    "n_attempts"    : kn["total_attempts"],
    "n_incomplete"  : kn["modules_incomplete"],
    "is_female"     : (kn["gender"] == "Female").astype(int),
})
yk = kn["y"].values
print("Source subset:", K.shape, "| at-risk", yk.mean().round(3))

Source subset: (3325, 9) | at-risk 0.17


In [17]:
# ---------- 2. OULAD side ----------
info = pd.read_csv("/content/oulad/studentInfo.csv")
info["y"] = (info.final_result == "Withdrawn").astype(int)
sa   = pd.read_csv("/content/oulad/studentAssessment.csv")
asmt = pd.read_csv("/content/oulad/assessments.csv")
sa = sa.merge(asmt[["id_assessment","code_module","code_presentation","assessment_type","date"]],
              on="id_assessment", how="left")
sa = sa[sa.assessment_type=="TMA"].sort_values(["id_student","code_module","code_presentation","date"])
KEY = ["id_student","code_module","code_presentation"]
sc_ = (sa.groupby(KEY).agg(score_mean=("score","mean"), score_sd=("score","std"),
        score_improve=("score", lambda x: float(x.iloc[-1]-x.iloc[0]) if len(x)>1 else 0.0),
        n_assess=("score","count")).reset_index())
sc_["score_sd"] = sc_["score_sd"].fillna(0)
eng = pd.read_parquet(f"{OUT}/oulad_engagement.parquet")
ou  = (info[KEY+["gender","num_of_prev_attempts","y"]]
        .merge(sc_, on=KEY, how="inner").merge(eng, on=KEY, how="inner"))
print("OULAD comparable:", len(ou), "| withdrawn", ou.y.mean().round(3))

O = pd.DataFrame({
    "eng_views_pct" : pct(ou["n_clicks"]),
    "eng_sess_pct"  : pct(ou["n_sessions"]),
    "eng_span_pct"  : pct(ou["engagement_span_days"]),
    "score_sd"      : ou["score_sd"],
    "score_mean"    : ou["score_mean"] / 10.0,       # 0-100 -> 0-10, Source module scale
    "score_improve" : ou["score_improve"],
    "n_attempts"    : ou["num_of_prev_attempts"],
    "n_incomplete"  : ou["n_assess"].max() - ou["n_assess"],   # inverse: missed assessments
    "is_female"     : (ou["gender"]=="F").astype(int),
})
yo = ou["y"].values
assert list(K.columns) == list(O.columns), "column order mismatch"

OULAD comparable: 24772 | withdrawn 0.168


In [18]:
# --- NaN handling, by type ---
# score_improve: one assessment -> no trajectory observed -> 0 is correct
O["score_improve"] = O["score_improve"].fillna(0)

# score_mean: all scores missing -> no assessment signal -> drop the record
keep = O["score_mean"].notna()
n_drop = int((~keep).sum())
O  = O[keep].reset_index(drop=True)
yo = yo[keep.values]

print(f"zero-filled score_improve : 100 (single-assessment students)")
print(f"dropped, no score data    : {n_drop} ({n_drop/len(keep):.2%})")
print(f"OULAD after cleaning      : {len(O):,} | withdrawn {yo.mean():.3f}")
assert O.isna().sum().sum() == 0, "NaNs remain in O"
assert K.isna().sum().sum() == 0, "NaNs in K"
print("clean — ready for Stage 1 model loop")

zero-filled score_improve : 100 (single-assessment students)
dropped, no score data    : 32 (0.13%)
OULAD after cleaning      : 24,740 | withdrawn 0.167
clean — ready for Stage 1 model loop


In [19]:
# ---------- 3. train Source-9, evaluate in-domain, then transfer ----------
def run(cols, drop_gender=False):
    use = [c for c in cols if not (drop_gender and c=="is_female")]
    in_dom, zero_shot = [], []
    for s in SEEDS:
        Xtr, Xte, ytr, yte = train_test_split(K[use], yk, test_size=0.20,
                                              stratify=yk, random_state=s)
        sc = MinMaxScaler(); Xtr_s = sc.fit_transform(Xtr); Xte_s = sc.transform(Xte)
        Xb, yb = SMOTE(random_state=s).fit_resample(Xtr_s, ytr)
        m = XGBClassifier(**BEST, eval_metric="auc", random_state=42).fit(Xb, yb)
        in_dom.append(roc_auc_score(yte, m.predict_proba(Xte_s)[:,1]))
        # ZERO-SHOT: same fitted model, same scaler, applied cold to OULAD
        zero_shot.append(roc_auc_score(yo, m.predict_proba(sc.transform(O[use]))[:,1]))
    return np.array(in_dom), np.array(zero_shot)

for label, dg in [("with gender", False), ("without gender", True)]:
    ind, zs = run(list(K.columns), drop_gender=dg)
    print(f"\n=== Source-9 {label} ===")
    print(f"  in-domain Source AUC : {ind.mean():.4f} ± {ind.std(ddof=1):.4f}")
    print(f"  zero-shot OULAD AUC : {zs.mean():.4f} ± {zs.std(ddof=1):.4f}")
    print(f"  transfer gap vs OULAD-from-scratch (0.937): {0.937-zs.mean():+.4f}")
    print(f"  per-seed zero-shot  : {np.round(zs,4)}")

print("\nReference points: Source 26-col deployed 0.870 | OULAD from-scratch 0.937 | chance 0.500")



=== Source-9 with gender ===
  in-domain Source AUC : 0.8559 ± 0.0063
  zero-shot OULAD AUC : 0.6423 ± 0.0473
  transfer gap vs OULAD-from-scratch (0.937): +0.2947
  per-seed zero-shot  : [0.6454 0.6193 0.7093 0.5812 0.6563]

=== Source-9 without gender ===
  in-domain Source AUC : 0.8564 ± 0.0079
  zero-shot OULAD AUC : 0.6198 ± 0.0394
  transfer gap vs OULAD-from-scratch (0.937): +0.3172
  per-seed zero-shot  : [0.6052 0.6178 0.6786 0.57   0.6276]

Reference points: Source 26-col deployed 0.870 | OULAD from-scratch 0.937 | chance 0.500


In [14]:
print("OULAD NaNs per column:")
print(O.isna().sum()[O.isna().sum() > 0])
print("\nSource NaNs per column:")
print(K.isna().sum()[K.isna().sum() > 0])
print("\nrows affected in O:", int(O.isna().any(axis=1).sum()), "of", len(O))

OULAD NaNs per column:
score_mean        32
score_improve    100
dtype: int64

Source NaNs per column:
Series([], dtype: int64)

rows affected in O: 130 of 24772


In [20]:
# TIER 3 — STAGE 2: Adaptation curve, Source -> OULAD
# Arm A: continued boosting from the Source booster (transfer value)
# Arm B: from-scratch on the same OULAD budget (data cost)
# Run AFTER stage 1 in the same session: K, yk, O, yo, BEST, SEEDS must exist.
# ============================================================
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

COLS    = list(K.columns)
BUDGETS = [100, 250, 500, 1000, 2500, 5000]   # OULAD training records made available
ADD_ROUNDS = 100                              # new trees appended in Arm A

# Fixed OULAD evaluation split, never trained on
O_tr, O_te, yo_tr, yo_te = train_test_split(
    O[COLS], yo, test_size=0.30, stratify=yo, random_state=1)
print(f"OULAD adapt pool {len(O_tr):,} | held-out eval {len(O_te):,} "
      f"(withdrawn {yo_te.mean():.3f})\n")

rows = []
for s in SEEDS:
    # --- refit the Source source model for this seed (as in Stage 1) ---
    Xtr, _, ytr, _ = train_test_split(K[COLS], yk, test_size=0.20,
                                      stratify=yk, random_state=s)
    sc = MinMaxScaler(); Xtr_s = sc.fit_transform(Xtr)
    Xb, yb = SMOTE(random_state=s).fit_resample(Xtr_s, ytr)
    src = XGBClassifier(**BEST, eval_metric="auc", random_state=42).fit(Xb, yb)

    O_te_s = sc.transform(O_te)                       # Source scaler = zero-shot condition
    zs = roc_auc_score(yo_te, src.predict_proba(O_te_s)[:, 1])
    rows.append(dict(seed=s, budget=0, arm="zero-shot", auc=zs))

    for n in BUDGETS:
        if n > len(O_tr): continue
        sub, _, ysub, _ = train_test_split(O_tr, yo_tr, train_size=n,
                                           stratify=yo_tr, random_state=s)
        # ---- Arm A: continue boosting from the Source trees ----
        sub_s = sc.transform(sub)                     # keep source scaler
        Xa, ya = SMOTE(random_state=s).fit_resample(sub_s, ysub)
        ft = XGBClassifier(**{**BEST, "n_estimators": ADD_ROUNDS},
                           eval_metric="auc", random_state=42)
        ft.fit(Xa, ya, xgb_model=src.get_booster())   # frozen source trees retained
        rows.append(dict(seed=s, budget=n, arm="fine-tuned",
                         auc=roc_auc_score(yo_te, ft.predict_proba(O_te_s)[:, 1])))

        # ---- Arm B: from scratch on the same budget, own scaler ----
        sc2 = MinMaxScaler(); sub_s2 = sc2.fit_transform(sub)
        Xc, yc = SMOTE(random_state=s).fit_resample(sub_s2, ysub)
        scratch = XGBClassifier(**BEST, eval_metric="auc", random_state=42).fit(Xc, yc)
        rows.append(dict(seed=s, budget=n, arm="from-scratch",
                         auc=roc_auc_score(yo_te, scratch.predict_proba(sc2.transform(O_te))[:, 1])))

df = pd.DataFrame(rows)
piv = df.groupby(["arm","budget"]).auc.agg(["mean","std"]).round(4)
print(piv.to_string(), "\n")

zs_m = df[df.arm=="zero-shot"].auc.mean()
print(f"{'budget':>8} {'fine-tuned':>18} {'from-scratch':>18} {'transfer benefit':>18}")
print(f"{0:>8} {zs_m:>18.4f} {'—':>18} {'—':>18}")
for n in BUDGETS:
    a = df[(df.arm=="fine-tuned")&(df.budget==n)].auc
    b = df[(df.arm=="from-scratch")&(df.budget==n)].auc
    if len(a):
        print(f"{n:>8} {a.mean():>10.4f} ±{a.std(ddof=1):.4f} "
              f"{b.mean():>10.4f} ±{b.std(ddof=1):.4f} {a.mean()-b.mean():>+18.4f}")

print(f"\nreference: zero-shot {zs_m:.4f} | full-data from-scratch 0.937")
df.to_csv("/content/drive/MyDrive/cinque_transfer_stage2.csv", index=False)
print("saved: cinque_transfer_stage2.csv")

OULAD adapt pool 17,318 | held-out eval 7,422 (withdrawn 0.167)

                       mean     std
arm          budget                
fine-tuned   100     0.8502  0.0396
             250     0.8947  0.0073
             500     0.8982  0.0081
             1000    0.9065  0.0059
             2500    0.9118  0.0030
             5000    0.9096  0.0039
from-scratch 100     0.8619  0.0460
             250     0.9005  0.0104
             500     0.9042  0.0070
             1000    0.9124  0.0025
             2500    0.9170  0.0020
             5000    0.9196  0.0018
zero-shot    0       0.6417  0.0467 

  budget         fine-tuned       from-scratch   transfer benefit
       0             0.6417                  —                  —
     100     0.8502 ±0.0396     0.8619 ±0.0460            -0.0117
     250     0.8947 ±0.0073     0.9005 ±0.0104            -0.0058
     500     0.8982 ±0.0081     0.9042 ±0.0070            -0.0060
    1000     0.9065 ±0.0059     0.9124 ±0.0025            -0.00

In [21]:
# ---- CONTROL: Arm A with a fresh OULAD scaler ----
# Isolates whether the negative transfer benefit comes from the source TREES
# or merely from the source SCALER's feature calibration.
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

COLS = list(K.columns)
rows = []

for s in SEEDS:
    # rebuild the same Source source model as before
    Xtr, _, ytr, _ = train_test_split(K[COLS], yk, test_size=0.20,
                                      stratify=yk, random_state=s)
    sc_src = MinMaxScaler(); Xtr_s = sc_src.fit_transform(Xtr)
    Xb, yb = SMOTE(random_state=s).fit_resample(Xtr_s, ytr)
    src = XGBClassifier(**BEST, eval_metric="auc", random_state=42).fit(Xb, yb)

    for n in BUDGETS:
        if n > len(O_tr): continue
        sub, _, ysub, _ = train_test_split(O_tr, yo_tr, train_size=n,
                                           stratify=yo_tr, random_state=s)
        # KEY DIFFERENCE: scaler refitted on the OULAD subset, not inherited
        sc_tgt = MinMaxScaler(); sub_s = sc_tgt.fit_transform(sub)
        Xa, ya = SMOTE(random_state=s).fit_resample(sub_s, ysub)
        ft = XGBClassifier(**{**BEST, "n_estimators": 100},
                           eval_metric="auc", random_state=42)
        ft.fit(Xa, ya, xgb_model=src.get_booster())
        rows.append(dict(seed=s, budget=n,
                         auc=roc_auc_score(yo_te, ft.predict_proba(sc_tgt.transform(O_te))[:, 1])))

ctl = pd.DataFrame(rows).groupby("budget").auc.agg(["mean","std"]).round(4)

print(f"{'budget':>7} {'A: src scaler':>15} {'A: tgt scaler':>15} {'from-scratch':>14} {'benefit(tgt)':>13}")
prev_ft = {100:0.8502, 250:0.8947, 500:0.8982, 1000:0.9065, 2500:0.9118, 5000:0.9096}
prev_fs = {100:0.8619, 250:0.9005, 500:0.9042, 1000:0.9124, 2500:0.9170, 5000:0.9196}
for n in BUDGETS:
    if n in ctl.index:
        m = ctl.loc[n,"mean"]
        print(f"{n:>7} {prev_ft[n]:>15.4f} {m:>15.4f} {prev_fs[n]:>14.4f} {m-prev_fs[n]:>+13.4f}")

 budget   A: src scaler   A: tgt scaler   from-scratch  benefit(tgt)
    100          0.8502          0.8611         0.8619       -0.0008
    250          0.8947          0.8968         0.9005       -0.0037
    500          0.8982          0.9000         0.9042       -0.0042
   1000          0.9065          0.9072         0.9124       -0.0052
   2500          0.9118          0.9110         0.9170       -0.0060
   5000          0.9096          0.9120         0.9196       -0.0076


In [22]:
# ============================================================
# ARCHIVE — Tier 3 transfer experiment
# Saves per-seed results, summaries, feature schema, and env metadata.
# Run in the same session as Stage 1 + Stage 2 + the scaler control.
# ============================================================
import os, json, sys, numpy as np, pandas as pd
import sklearn, xgboost, imblearn

OUT = "/content/drive/MyDrive/cinque_transfer"
os.makedirs(OUT, exist_ok=True)
PFX = f"{OUT}/cinque_transfer"

# ---------- 1. re-run Stage 1 capturing per-seed values ----------
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

COLS = list(K.columns)
s1 = []
for label, drop_g in [("with_gender", False), ("without_gender", True)]:
    use = [c for c in COLS if not (drop_g and c == "is_female")]
    for s in SEEDS:
        Xtr, Xte, ytr, yte = train_test_split(K[use], yk, test_size=0.20,
                                              stratify=yk, random_state=s)
        sc = MinMaxScaler(); Xtr_s = sc.fit_transform(Xtr)
        Xb, yb = SMOTE(random_state=s).fit_resample(Xtr_s, ytr)
        m = XGBClassifier(**BEST, eval_metric="auc", random_state=42).fit(Xb, yb)
        s1.append(dict(config=label, seed=s,
                       auc_in_domain=roc_auc_score(yte, m.predict_proba(sc.transform(Xte))[:, 1]),
                       auc_zero_shot=roc_auc_score(yo,  m.predict_proba(sc.transform(O[use]))[:, 1])))
s1 = pd.DataFrame(s1)
s1.to_csv(f"{PFX}_stage1_per_seed.csv", index=False)

s1_sum = s1.groupby("config")[["auc_in_domain","auc_zero_shot"]].agg(["mean","std"]).round(4)
s1_sum.to_csv(f"{PFX}_stage1_summary.csv")
print(s1_sum.to_string(), "\n")

# ---------- 2. Stage 2 (df already built by the Stage 2 cell) ----------
df.to_csv(f"{PFX}_stage2_per_seed.csv", index=False)
df.groupby(["arm","budget"]).auc.agg(["mean","std"]).round(4).to_csv(f"{PFX}_stage2_summary.csv")

# ---------- 3. scaler control (ctl from the control cell) ----------
try:
    ctl.to_csv(f"{PFX}_scaler_control.csv")
    ctl_ok = True
except NameError:
    ctl_ok = False
    print("NOTE: `ctl` not in memory — scaler control not archived")

# ---------- 4. feature schema, so the mapping is reproducible ----------
schema = pd.DataFrame({
    "transfer_feature": COLS,
    "source_source": ["n_module_views (pct)","n_logins (pct)","engagement_span_days (pct)",
                     "std_module_score","total_score / 6","score_improvement",
                     "total_attempts","modules_incomplete","gender == Female"],
    "oulad_source":  ["n_clicks (pct)","n_sessions (pct)","engagement_span_days (pct)",
                      "score_sd","score_mean / 10","score_improve",
                      "num_of_prev_attempts","max(n_assess) - n_assess","gender == F"],
})
schema.to_csv(f"{PFX}_feature_schema.csv", index=False)

# ---------- 5. run metadata ----------
meta = dict(
    experiment="Tier 3 zero-shot transfer + adaptation, Source -> OULAD",
    source_n=int(len(K)), source_at_risk_rate=float(np.mean(yk)),
    oulad_n=int(len(O)), oulad_withdrawn_rate=float(np.mean(yo)),
    n_transfer_features=len(COLS), transfer_features=COLS,
    seeds=SEEDS, hyperparameters=BEST,
    budgets=BUDGETS, added_boost_rounds=100,
    oulad_eval_holdout=int(len(O_te)),
    nan_policy="score_improve filled 0 (single assessment); 32 records dropped (no scores)",
    scaler_policy="Arm A inherits Source scaler; control refits on target subset",
    reference_points=dict(source_26col_deployed=0.870, oulad_from_scratch_full=0.937, chance=0.5),
    env=dict(python=sys.version.split()[0], sklearn=sklearn.__version__,
             xgboost=xgboost.__version__, imblearn=imblearn.__version__),
    scaler_control_archived=ctl_ok,
)
json.dump(meta, open(f"{PFX}_run_summary.json", "w"), indent=2, default=str)

print("saved to", OUT)
for f in sorted(os.listdir(OUT)):
    print(f"  {f}  ({os.path.getsize(os.path.join(OUT,f)):,} bytes)")

               auc_in_domain         auc_zero_shot        
                        mean     std          mean     std
config                                                    
with_gender           0.8559  0.0063        0.6423  0.0473
without_gender        0.8564  0.0079        0.6198  0.0394 

saved to /content/drive/MyDrive/cinque_transfer
  cinque_transfer_feature_schema.csv  (472 bytes)
  cinque_transfer_run_summary.json  (1,192 bytes)
  cinque_transfer_scaler_control.csv  (122 bytes)
  cinque_transfer_stage1_per_seed.csv  (585 bytes)
  cinque_transfer_stage1_summary.csv  (170 bytes)
  cinque_transfer_stage2_per_seed.csv  (2,501 bytes)
  cinque_transfer_stage2_summary.csv  (407 bytes)
